============================================================
[프로젝트] 2026년 2차 서울시 청년안심주택(공공임대) 청약 도우미 RAG
============================================================

In [1]:
import os
from dotenv import load_dotenv
 
load_dotenv()
 
# =========================================================
# 0. 환경변수 확인
# =========================================================
if os.environ.get("OPENAI_API_KEY"):
    print("✓ OpenAI API Key가 설정되었습니다.")
else:
    print("✗ OpenAI API Key가 없습니다.")
 
if os.environ.get("QDRANT_URL") and os.environ.get("QDRANT_API_KEY"):
    print("✓ Qdrant Cloud 설정이 완료되었습니다.")
    print(f"  URL: {os.environ.get('QDRANT_URL')}")
else:
    print("✗ Qdrant Cloud 설정이 필요합니다.")
    print("  .env 파일에 QDRANT_URL과 QDRANT_API_KEY를 추가하세요.")

✓ OpenAI API Key가 설정되었습니다.
✓ Qdrant Cloud 설정이 완료되었습니다.
  URL: https://cb5cd2b9-a3d0-41be-af57-e7885163299f.us-east-1-1.aws.cloud.qdrant.io:6333


In [2]:
# =========================================================
# 1. 프로젝트 주제 & Use Case 정의
# =========================================================
# 서비스 주제: "2026년 2차 서울시 청년안심주택(공공임대) 청약 도우미 챗봇"
 
USE_CASES = [
    "청약 일정 문의",
    "신청자격 확인",
    "단지별 임대조건 조회",
    "가점 계산",
    "제출서류 안내",
    "셰어형(쉐어하우스) 청약방법",
    "소득·자산 기준 판단",
    "재계약·거주기간 규정",
    "단지별 위치·시공사 정보 조회",
    "입주 후 생활 관련 유의사항",
]
 
SAMPLE_QUESTIONS = [
    "청약 접수는 언제부터 언제까지인가요?",
    "39세인데 청년계층 신청 가능한가요?",
    "에이트플레이스 39A 타입 임대보증금과 월세는 얼마인가요?",
    "청약통장 24회 납입하고 서울 거주 5년이면 가점이 몇 점인가요?",
    "청년 2순위로 신청하려면 어떤 서류를 준비해야 하나요?",
    "2인 1팀으로 셰어형 신청하려면 어떻게 하나요?",
    "3인가구 월평균소득 500만원이면 신혼부부Ⅰ 몇 % 구간인가요?",
    "청년으로 입주 후 결혼하면 거주기간이 어떻게 되나요?",
    "용산 원효 루미니 시공사가 어디인가요?",
    "반려동물 키울 수 있나요?",
]

In [3]:
# =========================================================
# 2. PDF 문서 로드 (PyMuPDF / fitz 사용)
# =========================================================
# pypdf 설치 문제를 피하기 위해 fitz(PyMuPDF)로 텍스트를 읽고,
# 이후 단계(청킹/메타데이터/Qdrant 적재)와 호환되도록
# LangChain의 Document 객체(.page_content, .metadata)로 감싸준다.
import fitz  # PyMuPDF
from langchain_core.documents import Document
 
PDF_PATH = "./공공_260731_2026년 2차 청년안심주택 모집공고문.pdf"
 
# 경로 문제를 미리 확인 (파일이 없으면 여기서 원인을 바로 알려줌)
if not os.path.exists(PDF_PATH):
    folder = os.path.dirname(PDF_PATH) or "."
    print(f"✗ 파일을 찾을 수 없습니다: {PDF_PATH}")
    print(f"  현재 작업 디렉토리: {os.getcwd()}")
    if os.path.exists(folder):
        print(f"  '{folder}' 폴더 안의 실제 파일 목록:")
        for f in os.listdir(folder):
            print(f"    - {f}")
    else:
        print(f"  '{folder}' 폴더 자체가 존재하지 않습니다.")
    raise FileNotFoundError(PDF_PATH)
 
pdf_doc = fitz.open(PDF_PATH)
 
docs = []
for i, page in enumerate(pdf_doc):
    text = page.get_text("text", sort=True)
    docs.append(
        Document(
            page_content=text,
            metadata={"page": i, "source": PDF_PATH},
        )
    )
 
pdf_doc.close()
 
print(f"✓ PDF 로드 완료: 총 {len(docs)} 페이지")
print(f"  예시(0페이지 앞부분): {docs[0].page_content[:80]}...")

✓ PDF 로드 완료: 총 68 페이지
  예시(0페이지 앞부분):                         www.i-sh.co.kr





2026년 2차 서울시 청년안심주택(공공임대)
 입주자 모집공고
...


In [4]:
# =========================================================
# 3. 문서 분할(Chunking)
# =========================================================
from langchain_text_splitters import RecursiveCharacterTextSplitter
 
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
)
 
split_docs = text_splitter.split_documents(docs)
print(f"✓ 청킹 완료: {len(docs)}페이지 → {len(split_docs)}개 청크")
 
 
# =========================================================
# 3-1. 메타데이터 부여 (카테고리 분류)
# =========================================================
# 목차(1~14장) 기준 페이지 구간별 category 부여
# → 5단계 동적 필터링에서 사용
# ※ 실제 PDF의 page 번호와 맞는지 확인 후 range 조정하세요.
 
CATEGORY_BY_PAGE = {
    range(0, 6):   "공급일정",
    range(6, 26):  "공급현황",
    range(26, 29): "임대조건",
    range(29, 41): "신청자격",
    range(41, 43): "신청접수",
    range(43, 49): "제출서류",
    range(49, 51): "당첨자발표",
    range(51, 52): "계약입주",
    range(52, 54): "거주기간",
    range(54, 56): "신청유의사항",
    range(56, 59): "단지유의사항",
    range(59, 67): "추가안내",
}
 
 
def get_category(page_num: int) -> str:
    for page_range, category in CATEGORY_BY_PAGE.items():
        if page_num in page_range:
            return category
    return "기타"
 
 
for doc in split_docs:
    page = doc.metadata.get("page", 0)
    doc.metadata["category"] = get_category(page)
    doc.metadata["source"] = "2026년_2차_청년안심주택_모집공고문"
 
print("✓ 메타데이터(category) 부여 완료")
print(f"  예시: {split_docs[0].metadata}")

✓ 청킹 완료: 68페이지 → 363개 청크
✓ 메타데이터(category) 부여 완료
  예시: {'page': 0, 'source': '2026년_2차_청년안심주택_모집공고문', 'category': '공급일정'}


In [5]:
# =========================================================
# 4. Qdrant VectorDB 적재
# =========================================================
from langchain_openai import OpenAIEmbeddings
from langchain_qdrant import QdrantVectorStore
 
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
 
COLLECTION_NAME = "cheongnyeon_anshim_housing"
 
# from_documents 한 줄로 컬렉션 생성 + 임베딩 + 업로드까지 처리됨
vectorstore = QdrantVectorStore.from_documents(
    split_docs,
    embeddings,
    url=os.environ.get("QDRANT_URL"),
    api_key=os.environ.get("QDRANT_API_KEY"),
    collection_name=COLLECTION_NAME,
)
 
print(f"✓ Qdrant 적재 완료: 컬렉션 '{COLLECTION_NAME}'에 {len(split_docs)}개 청크 업로드")
 
# Qdrant는 filter(models.FieldCondition)로 검색할 필드에 대해
# 사전에 payload 인덱스가 만들어져 있어야 함 (없으면 400 Bad Request)
# → 6단계에서 metadata.category로 필터링하므로 여기서 미리 인덱스 생성
vectorstore.client.create_payload_index(
    collection_name=COLLECTION_NAME,
    field_name="metadata.category",
    field_schema="keyword",
)
print("✓ 'metadata.category' 필드 인덱스 생성 완료")
 
 
# =========================================================
# 4-1. (선택/심화) Parent Document Retriever
# =========================================================
# 검색은 잘게 쪼갠 child 청크로 정밀하게 하고,
# 답변 생성에 쓸 컨텍스트는 더 큰 parent 청크(원문 맥락)를 가져오는 방식
# → 과제 요구사항이라 별도 블록으로 최소 구현만 붙여둠
 
from langchain_core.stores import InMemoryStore
from langchain_classic.retrievers import ParentDocumentRetriever
 
parent_splitter = RecursiveCharacterTextSplitter(chunk_size=1500, chunk_overlap=100)
child_splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=50)
 
# Parent Document Retriever 전용 컬렉션 (위 vectorstore와 별도로 하나 더 사용)
parent_vectorstore = QdrantVectorStore.from_documents(
    [],  # 빈 상태로 시작 (add_documents로 나중에 채움)
    embeddings,
    url=os.environ.get("QDRANT_URL"),
    api_key=os.environ.get("QDRANT_API_KEY"),
    collection_name=f"{COLLECTION_NAME}_parent",
)
 
docstore = InMemoryStore()
 
parent_document_retriever = ParentDocumentRetriever(
    vectorstore=parent_vectorstore,
    docstore=docstore,
    child_splitter=child_splitter,
    parent_splitter=parent_splitter,
)
 
parent_document_retriever.add_documents(docs)  # 원본 페이지 단위 docs 사용
print(f"✓ Parent Document Retriever 적재 완료 (parent 문서 수: {len(list(docstore.yield_keys()))})")

✓ Qdrant 적재 완료: 컬렉션 'cheongnyeon_anshim_housing'에 363개 청크 업로드
✓ 'metadata.category' 필드 인덱스 생성 완료
✓ Parent Document Retriever 적재 완료 (parent 문서 수: 123)


In [6]:
# =========================================================
# 5. 검색(Retrieval) 테스트
# =========================================================
print("\n" + "=" * 60)
print("검색(Retrieval) 테스트 - Top-k 확인")
print("=" * 60)
 
for question in SAMPLE_QUESTIONS[:5]:
    print(f"\n[질문] {question}")
    results = vectorstore.similarity_search(question, k=3)
    for i, doc in enumerate(results, 1):
        category = doc.metadata.get("category", "기타")
        page = doc.metadata.get("page", "?")
        preview = doc.page_content.replace("\n", " ")[:150]
        print(f"  ({i}) [category={category}, page={page}] {preview}...")


검색(Retrieval) 테스트 - Top-k 확인

[질문] 청약 접수는 언제부터 언제까지인가요?
  (1) [category=공급일정, page=5] ※ 일정은 진행상황에 따라 변동 가능하며, 입주지정기간은 입주 가능한날로부터 30일 간  ■ 인터넷 청약신청 세부일정      대상자                접수일정              인터넷청약 주소      청년계층,                       ...
  (2) [category=공급일정, page=5] ※ 일정은 진행상황에 따라 변동 가능하며, 입주지정기간은 입주 가능한날로부터 30일 간  ■ 인터넷 청약신청 세부일정      대상자                접수일정              인터넷청약 주소      청년계층,                       ...
  (3) [category=공급일정, page=5] ※ 일정은 진행상황에 따라 변동 가능하며, 입주지정기간은 입주 가능한날로부터 30일 간  ■ 인터넷 청약신청 세부일정      대상자                접수일정              인터넷청약 주소      청년계층,                       ...

[질문] 39세인데 청년계층 신청 가능한가요?
  (1) [category=신청자격, page=29] ■ 신청유형  ○ 아래의 신청 유형 중 모집공고일 현재 신청자 본인에 해당하는 유형 한 가지를 선택하여 신청해야 합니다.  ○ 순위별 소득 및 자산기준 등을 충족하더라도, 신청유형 관련 세부기준을 충족하지 못하면 서류심사 시 입주자격  부적격으로 판정되므로 세부기준을 ...
  (2) [category=신청자격, page=29] ■ 신청유형  ○ 아래의 신청 유형 중 모집공고일 현재 신청자 본인에 해당하는 유형 한 가지를 선택하여 신청해야 합니다.  ○ 순위별 소득 및 자산기준 등을 충족하더라도, 신청유형 관련 세부기준을 충족하지 못하면 서류심사 시 입주자격  부적격으로 판정되므로 세부기준

In [7]:
# =========================================================
# 6. (시간 여유시) 메타데이터 필터링 + RAG 답변 생성
# =========================================================
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate
from qdrant_client.http import models
 
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
 
# 질문 속 키워드 → 카테고리 매핑 (동적 필터링용)
KEYWORD_TO_CATEGORY = {
    "일정": "공급일정", "접수": "공급일정", "언제": "공급일정",
    "자격": "신청자격", "순위": "신청자격", "무주택": "신청자격",
    "임대료": "임대조건", "보증금": "임대조건", "월세": "임대조건",
    "가점": "신청자격", "점수": "신청자격",
    "서류": "제출서류", "제출": "제출서류",
    "당첨": "당첨자발표", "발표": "당첨자발표",
    "계약": "계약입주", "입주": "계약입주",
    "거주기간": "거주기간", "갱신": "거주기간", "재계약": "거주기간",
    "유의사항": "신청유의사항",
    "반려동물": "단지유의사항", "차량": "단지유의사항", "주차": "단지유의사항",
    "시공사": "공급현황", "세대": "공급현황",
    "소득": "추가안내", "자산": "추가안내", "금융": "추가안내",
}
 
 
def determine_category(question: str):
    for keyword, category in KEYWORD_TO_CATEGORY.items():
        if keyword in question:
            return category
    return None
 
 
def rag_with_dynamic_filter(question: str) -> str:
    """
    동적 필터링을 적용한 RAG
    """
    # 1. 질문 분석하여 카테고리 결정
    category = determine_category(question)
 
    # 2. 필터 설정
    search_kwargs = {"k": 3}
    if category:
        search_kwargs["filter"] = models.Filter(
            must=[
                models.FieldCondition(
                    key="metadata.category",
                    match=models.MatchValue(value=category),
                )
            ]
        )
        print(f"✓ 적용된 필터: category = '{category}'\n")
    else:
        print("✓ 필터 없음 (전체 문서 검색)\n")
 
    # 3. 문서 검색
    retriever = vectorstore.as_retriever(search_kwargs=search_kwargs)
    retrieved_docs = retriever.invoke(question)
 
    # 4. 컨텍스트 구성
    context_parts = []
    for doc in retrieved_docs:
        page = doc.metadata.get("page", "?")
        cat = doc.metadata.get("category", "기타")
        context_parts.append(
            f"[출처: {doc.metadata.get('source', '청년안심주택 모집공고문')}, 페이지: {page}, 카테고리: {cat}]\n{doc.page_content}"
        )
 
    context = "\n\n---\n\n".join(context_parts)
 
    # 5. 프롬프트 생성
    template = """
당신은 2026년 2차 서울시 청년안심주택(공공임대) 청약 안내 전문가입니다.
주어진 정보를 바탕으로 사용자의 질문에 정확하고 친절하게 답변하세요.
답변에 참고한 문서의 출처와 페이지 번호를 명시하세요.
 
<context>
{context}
</context>
 
<question>
{question}
</question>
"""
 
    prompt_template = PromptTemplate(
        input_variables=["context", "question"],
        template=template,
    )
 
    formatted_prompt = prompt_template.format(context=context, question=question)
 
    # 6. LLM 호출
    response = llm.invoke(formatted_prompt)
    return response.content
 
 
# 테스트 실행
for question in SAMPLE_QUESTIONS:
    print(f"\n질문: {question}\n")
    answer = rag_with_dynamic_filter(question)
    print(f"답변:\n{answer}\n")
    print("-" * 60)


질문: 청약 접수는 언제부터 언제까지인가요?

✓ 적용된 필터: category = '공급일정'

답변:
청약 접수는 2026년 8월 11일(화) 10:00부터 2026년 8월 13일(목) 17:00까지입니다. 청약 신청은 인터넷을 통해서만 가능하며, 신청 주소는 www.i-sh.co.kr/app입니다. (출처: 2026년_2차_청년안심주택_모집공고문, 페이지: 5)

------------------------------------------------------------

질문: 39세인데 청년계층 신청 가능한가요?

✓ 필터 없음 (전체 문서 검색)

답변:
네, 39세이신 경우 청년계층으로 신청이 가능합니다. 청년안심주택의 신청자격은 19세 이상 39세 이하의 미혼 청년으로, 직장 재직 여부와 관계없이 대학생 및 취업준비생도 지원할 수 있습니다. 따라서 39세이신 분도 신청이 가능합니다. 

자세한 내용은 2026년 2차 청년안심주택 모집공고문에서 확인하실 수 있으며, 해당 정보는 페이지 29에 기재되어 있습니다.

------------------------------------------------------------

질문: 에이트플레이스 39A 타입 임대보증금과 월세는 얼마인가요?

✓ 적용된 필터: category = '임대조건'

답변:
2026년 2차 서울시 청년안심주택의 에이트플레이스 39A 타입의 임대보증금과 월세에 대한 구체적인 정보는 제공된 문서에 포함되어 있지 않습니다. 해당 정보는 서울시 청년안심주택의 공식 웹사이트나 주거안심종합센터에 문의하시면 확인하실 수 있습니다.

추가적으로, 임대조건에 대한 정보는 문서의 페이지 28에서 확인할 수 있습니다. 만약 다른 질문이 있으시면 언제든지 말씀해 주세요!

------------------------------------------------------------

질문: 청약통장 24회 납입하고 서울 거주 5년이면 가점이 몇 점인가요?

✓ 적용된 필터: category = '